# RAG Embedding Generator — Genomics Edition

Upload **8 zip files** (one per sample). Each zip is expected to contain:

| File pattern | Content |
|---|---|
| `quality_indicators_*.txt` | QC metrics — pass/fail thresholds |
| `full_variant_table_*.txt` | SNPs/INDELs with ClinVar, gnomAD, SIFT, PolyPhen |
| `full_cnv_table_*.txt` | Structural variants / CNVs |
| `gene_cnv_table_*.txt` | Gene-level CNV + OMIM disease links |
| `flagged_regions_*.txt` | Low-coverage / problematic regions |
| `exon_coverage_stats_*.txt` | Per-exon depth statistics |
| `coverage_1bp_*.txt` | Base-pair resolution coverage (skipped — raw positional data, not useful for RAG) |
| `OMIM_annotation_table_*.txt` | Gene → disease associations |

Each zip produces one `<zipname>.parquet` file. Download them all and load locally with `load-embeddings.py`.

> **Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── 1. Install deps ───────────────────────────────────────────────────────────
!pip install -q sentence-transformers pandas pyarrow

In [ ]:
# ── 2. Config ─────────────────────────────────────────────────────────────────
import torch

MODEL_NAME    = 'sentence-transformers/all-mpnet-base-v2'  # 768-dim
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

# Files to skip entirely (raw positional data — too large, no semantic value for RAG)
SKIP_PATTERNS = ['coverage_1bp_']

# Per-file-type chunking: (chunk_size_chars, overlap_chars)
# Larger chunks for dense annotation tables so each chunk has full row context.
CHUNK_PROFILES = {
    'quality_indicators_':    (600,  60),   # short metric rows, keep together
    'full_variant_table_':    (800,  80),   # rich annotation rows, need more context
    'full_cnv_table_':        (700,  70),
    'gene_cnv_table_':        (700,  70),
    'flagged_regions_':       (500,  50),
    'exon_coverage_stats_':   (600,  60),
    'OMIM_annotation_table_': (600,  60),
}
DEFAULT_CHUNK = (500, 50)

print(f'Device: {DEVICE}')

In [ ]:
# ── 3. Upload zip files ───────────────────────────────────────────────────────
from google.colab import files as colab_files

uploaded = colab_files.upload()   # select all 8 .zip files at once
zip_names = [name for name in uploaded if name.endswith('.zip')]
print(f'Uploaded {len(zip_names)} zip(s):', zip_names)

In [ ]:
# ── 4. Extract + chunk ────────────────────────────────────────────────────────
import io, zipfile, pathlib

def should_skip(filename):
    return any(pat in filename for pat in SKIP_PATTERNS)

def get_chunk_profile(filename):
    for pat, profile in CHUNK_PROFILES.items():
        if pat in filename:
            return profile
    return DEFAULT_CHUNK

def chunk_text(text, size, overlap):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size].strip())
        start += size - overlap
    return [c for c in chunks if c]

# zip_name -> list of {source, content} dicts
zip_rows = {}

for zip_name in zip_names:
    rows = []
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as zf:
        txt_files = [n for n in zf.namelist() if n.endswith('.txt')]
        print(f'\n{zip_name}: {len(txt_files)} txt files')
        for txt_name in sorted(txt_files):
            basename = pathlib.Path(txt_name).name
            if should_skip(basename):
                print(f'  skip  {basename}')
                continue
            size, overlap = get_chunk_profile(basename)
            text = zf.read(txt_name).decode('utf-8', errors='replace')
            chunks = chunk_text(text, size, overlap)
            for chunk in chunks:
                rows.append({'source': basename, 'content': chunk})
            print(f'  {basename}: {len(chunks)} chunks  (profile {size}/{overlap})')
    zip_rows[zip_name] = rows
    print(f'  → {len(rows)} total chunks for {zip_name}')

In [ ]:
# ── 5. Load model ─────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f'Model loaded on {DEVICE}')

In [ ]:
# ── 6. Embed + export one parquet per zip ─────────────────────────────────────
import pandas as pd

output_files = []

for zip_name, rows in zip_rows.items():
    if not rows:
        print(f'No rows for {zip_name}, skipping.')
        continue

    stem = pathlib.Path(zip_name).stem   # e.g. "SG063-LPA"
    out  = f'{stem}.parquet'

    print(f'\nEmbedding {len(rows)} chunks for {zip_name} …')
    texts = [r['content'] for r in rows]
    embeddings = model.encode(
        texts,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

    df = pd.DataFrame(rows)
    df['embedding'] = [e.tolist() for e in embeddings]
    df.to_parquet(out, index=False)

    mb = pathlib.Path(out).stat().st_size / 1e6
    print(f'  Saved {out}  ({mb:.1f} MB,  shape {df.shape})')
    output_files.append(out)

print(f'\nAll done. {len(output_files)} parquet file(s) ready to download.')

In [ ]:
# ── 7. Download all parquets ──────────────────────────────────────────────────
for out in output_files:
    print(f'Downloading {out} …')
    colab_files.download(out)

print()
print('Move each .parquet into backend/ingestion/ and run:')
print('  uv run python load-embeddings.py <file>.parquet')